In [1]:
import os
import json
import clip
from torchvision.datasets import ImageNet, Flowers102, Food101, ImageFolder
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, roc_curve
import numpy as np
from scipy.special import softmax
from tqdm import tqdm
from scipy.stats import multivariate_normal
from sklearn.decomposition import PCA
import pickle

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
clip_name = 'ViT-B/32' # 'ViT-B/32' or 'ViT-B/16
clip_path = clip_name.replace('/', '_')
dataset = 'Food101' # 'ImageNet', 'Flowers102', 'Food101', 'EUROSAT', 'DTD'

# Create Model and DataLoader

In [5]:
model, transform = clip.load(clip_name, device=DEVICE)

In [6]:
SRC_PATH = '/home/zhenxiang/Research/tmpDatasets'
# Test dataset
if dataset == 'ImageNet':
    test_dataset = ImageNet(root=f'{SRC_PATH}/ImageNet', split='val', transform=transform)
    rag_dataset = ImageNet(root=f'{SRC_PATH}/ImageNet', split='train', transform=transform)
    class_list = json.load(open(f'imagenet_labels.json'))
elif dataset == 'Food101':
    test_dataset = Food101(root=f'{SRC_PATH}/Food101', split='test', transform=transform, download=True)
    rag_dataset = Food101(root=f'{SRC_PATH}/Food101', split='train', transform=transform, download=True)
    class_list = test_dataset.classes
elif dataset == 'Flowers102':
    test_dataset = Flowers102(root=f'{SRC_PATH}/Flowers102', split='test', transform=transform, download=True)
    rag_dataset = Flowers102(root=f'{SRC_PATH}/Flowers102', split='train', transform=transform, download=True)
    class_list = test_dataset.classes
elif dataset == 'EUROSAT':
    test_dataset = ImageFolder(f'{SRC_PATH}/EUROSAT/test', transform=transform)
    rag_dataset = ImageFolder(f'{SRC_PATH}/EUROSAT/train', transform=transform)
    class_list = ['Annual Crop Land', 'Forest', 'Herbaceous Vegetation Land', 'Highway or Road', 'Industrial Buildings', 'Pasture Land', 'Permanent Crop Land', 'Residential Buildings', 'River', 'Sea or Lake']
elif dataset == 'DTD':
    test_dataset = ImageFolder(root=f'{SRC_PATH}/DTD/test', transform=transform)
    rag_dataset = ImageFolder(root=f'{SRC_PATH}/DTD/train', transform=transform)
    class_list = test_dataset.classes
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=16)
rag_loader = DataLoader(rag_dataset, batch_size=64, shuffle=False, num_workers=16)
dataset, len(test_dataset), len(rag_dataset)

('Food101', 25250, 75750)

# Extract Text Feature

In [7]:
token_text = [f'a photo of a {x}.' for x in class_list]
with torch.no_grad():
    clip_text = clip.tokenize(token_text).to(DEVICE)
    text_features = model.encode_text(clip_text)
    text_features = text_features / text_features.norm(dim=1, keepdim=True)
text_features.shape

torch.Size([101, 512])

# Test baseline

# Get External RAG Database and calculate Gaussian Distribution and PCA

In [8]:
@torch.no_grad()
def gaussian_rag():
    pca = PCA(n_components=128)
    base_path = os.path.join('pca-model', clip_path, dataset)
    os.makedirs(base_path, exist_ok=True)
    if not (os.path.exists(os.path.join(base_path, 'all_image_features.pt')) and os.path.exists(os.path.join(base_path, 'all_rag_targets.npy'))):
        all_image_features = []
        all_rag_targets = []
        for i, (images, target) in enumerate(tqdm(rag_loader)):
            images = images.to(DEVICE)
            target = target.to(DEVICE)
            image_features = model.encode_image(images)
            all_image_features.append(image_features.cpu())
            all_rag_targets.append(target.cpu().numpy())

        all_image_features = torch.cat(all_image_features, axis=0)
        all_rag_targets = np.concatenate(all_rag_targets, axis=0)
        torch.save(all_image_features, os.path.join(base_path, 'all_image_features.pt'))
        np.save(os.path.join(base_path, 'all_rag_targets.npy'), all_rag_targets)
    else:
        all_image_features = torch.load(os.path.join(base_path, 'all_image_features.pt'))
        all_rag_targets = np.load(os.path.join(base_path, 'all_rag_targets.npy'))

    if not os.path.exists(os.path.join(base_path, 'pca_model.pkl')):
        all_image_features = pca.fit_transform(all_image_features.numpy())
        with open(os.path.join(base_path, 'pca_model.pkl'), 'wb') as f:
            pickle.dump(pca, f)
    else:
        with open(os.path.join(base_path, 'pca_model.pkl'), 'rb') as f:
            pca = pickle.load(f)
        all_image_features = pca.transform(all_image_features.numpy())

    if not os.path.exists(os.path.join(base_path, 'gaussian_model.pkl')):
        gaussian_model = []
        for i in tqdm(range(len(class_list))):
            index = np.where(all_rag_targets == i)
            feat_i = all_image_features[index]
            mean_vec = np.mean(feat_i, axis=0)
            cov_mat = np.cov(feat_i, rowvar=False)
            cov_mat += np.eye(cov_mat.shape[0]) * 1e-8
            gaussian_model_i = multivariate_normal(mean=mean_vec, cov=cov_mat)
            gaussian_model.append(gaussian_model_i)
        with open(os.path.join(base_path, 'gaussian_model.pkl'), 'wb') as f:
            pickle.dump(gaussian_model, f)
    else:
        with open(os.path.join(base_path, 'gaussian_model.pkl'), 'rb') as f:
            gaussian_model = pickle.load(f)
    return pca, gaussian_model

pca, gaussian_model = gaussian_rag()

100%|██████████| 101/101 [00:01<00:00, 93.83it/s]


# Evaluate by RAG logPDF

In [9]:
def batched_multivariate_logpdf_torch(X, means, covs):
    """
    Args:
    X     : (N, d)       - N input samples
    means : (M, d)       - M gaussian distributions' means
    covs  : (M, d, d)    - M gaussian distributions' covariance matrices (must be positive definite)
    ---------------------------------------------------------------------------
    Return:
    logpdf: (N, M)       - the i-th row and j-th column represent the logpdf of X[i] under the j-th gaussian distribution
    """
    N, d = X.shape
    M = means.shape[0]

    # boardcast dev = X - means，get (N, M, d)
    dev = X[:, None, :] - means[None, :, :]  # (N, M, d)

    # Calculate the inverse (M, d, d) and logdet (M) of each covariance matrix
    L = torch.linalg.cholesky(covs) # A more stable way to handle positive definite matrices
    log_det = 2 * torch.sum(torch.log(torch.diagonal(L, dim1=1, dim2=2)), dim=1)  # (M,)
    inv_covs = torch.cholesky_inverse(L)          # (M, d, d)

    # calculate the squared Mahalanobis distance: dev @ inv_covs @ dev^T, output (N, M)
    mahal = torch.einsum('nmd,mdk,nmk->nm', dev, inv_covs, dev)  # (N, M)

    # normalization term
    log_norm = -0.5 * (d * torch.log(torch.tensor(2 * torch.pi)) + log_det)  # (M,)

    # combine
    logpdf = log_norm[None, :] - 0.5 * mahal  # (N, M)
    return logpdf

In [10]:
@torch.no_grad()
def test_rag(gaussian_model, use_softmax=False, use_diag=False):
    gaussian_mean = [g.mean for g in gaussian_model]
    if use_diag:
        gaussian_cov = [g.cov*np.eye(g.cov.shape[0]) for g in gaussian_model]
    else:
        gaussian_cov = [g.cov for g in gaussian_model]
    gaussian_mean = np.array(gaussian_mean)
    gaussian_cov = np.array(gaussian_cov)

    gaussian_mean = torch.tensor(gaussian_mean, device=DEVICE)
    gaussian_cov = torch.tensor(gaussian_cov, device=DEVICE)

    
    logit_scale = model.logit_scale.exp()
    all_pdf = []
    all_cosine = []
    gt_labels = []

    for i, (images, target) in enumerate(tqdm(test_loader)):
        images = images.to(DEVICE)
        target = target.to(DEVICE)
        image_features = model.encode_image(images)  
        image_features_pca = pca.transform(image_features.cpu().numpy())
        pdf = batched_multivariate_logpdf_torch(torch.tensor(image_features_pca, device=DEVICE), gaussian_mean, gaussian_cov)
        image_features_norm = image_features / image_features.norm(dim=1, keepdim=True)
        logits_per_image = logit_scale * (image_features_norm @ text_features.t())

        all_pdf.append(pdf.cpu().numpy())
        gt_labels.append(target.cpu().numpy())
        all_cosine.append(logits_per_image.cpu().numpy())
    gt_labels = np.concatenate(gt_labels, axis=0)
    all_cosine = np.concatenate(all_cosine, axis=0)
    all_pdf = np.concatenate(all_pdf, axis=0)

    print(f"gt_labels shape: {gt_labels.shape}, all_pdf shape: {all_pdf.shape}, all_cosine shape: {all_cosine.shape}")
    top_1_index = np.argmax(all_cosine, axis=1)

    if use_softmax:
        max_pdf = softmax(all_pdf, axis=1)[np.arange(all_cosine.shape[0]), top_1_index]
    else:
        max_pdf = all_pdf[np.arange(all_cosine.shape[0]), top_1_index]
    
    all_softmax = softmax(all_cosine, axis=1)
    max_softmax = all_softmax[np.arange(all_cosine.shape[0]), top_1_index]
    
    max_score = max_softmax + max_pdf
    max_score = 0.5 * max_score

    binary_gt_label = (gt_labels == top_1_index).astype(int)
    auroc = roc_auc_score(binary_gt_label, max_score) * 100
    print(f"AuROC: {auroc:.2f}")
    precision, recall, _ = precision_recall_curve(binary_gt_label, max_score)
    auc_pr = auc(recall, precision) * 100
    print(f"AuPR: {auc_pr:.2f}")  
    fpr, tpr, thresholds = roc_curve(binary_gt_label, max_score)
    idx_tpr_95 = np.argmin(np.abs(tpr - 0.95))
    fpr_in_tpr_95 = fpr[idx_tpr_95] * 100
    print(f"FPR at TPR=0.95: {fpr_in_tpr_95:.2f}")
    acc = (gt_labels == top_1_index).mean()
    print(f"Acc: {acc:.4f}")

test_rag(gaussian_model, use_softmax=True, use_diag=False)


  0%|          | 0/395 [00:00<?, ?it/s]

100%|██████████| 395/395 [00:10<00:00, 37.96it/s]


gt_labels shape: (25250,), all_pdf shape: (25250, 101), all_cosine shape: (25250, 101)
AuROC: 92.58
AuPR: 98.08
FPR at TPR=0.95: 33.98
Acc: 0.8122
